In [2]:
import subprocess
import sys

packages = [
    "langchain",
    "langchain-community",
    "chromadb",
    "sentence-transformers",
    "pymupdf",
    "unstructured",
    "beautifulsoup4",
    "groq",
    "python-dotenv"
]

for package in packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

print("all good")

all good


In [3]:
import json
import os

# path setting
BASE_DIR = "/Users/bella/powertrust_rag"
METADATA_PATH = os.path.join(BASE_DIR, "data/malaysia/Metadata/metadata_master.json")
DATA_DIR = os.path.join(BASE_DIR, "data/malaysia")
CHROMA_DIR = os.path.join(BASE_DIR, "chroma_db")

# read metadata_master.json
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata_master = json.load(f)

print(f"metadata_master.json successfully loaded")
print(f"Total top-level keys: {len(metadata_master)}")
print(f"Keys: {list(metadata_master.keys())}")

metadata_master.json successfully loaded
Total top-level keys: 6
Keys: ['part1_cost_economics', 'part2_grid_access', 'part3_policy', 'part4_utility', 'part5_approval', 'part6_unknown']


In [4]:
# Define chunk sizes by doc_type
CHUNK_CONFIG = {
    # Regulatory / guideline documents
    "regulatory_guidelines": {"chunk_size": 800, "overlap": 100},
    "contract_template":     {"chunk_size": 800, "overlap": 100},
    "technical_regulation":  {"chunk_size": 800, "overlap": 100},
    "technical_guideline":   {"chunk_size": 800, "overlap": 100},
    "process_flowchart":     {"chunk_size": 800, "overlap": 100},
    "application_form":      {"chunk_size": 800, "overlap": 100},

    # Research / report documents
    "research_report":       {"chunk_size": 1000, "overlap": 150},
    "annual_report":         {"chunk_size": 1000, "overlap": 150},
    "country_report":        {"chunk_size": 1000, "overlap": 150},
    "policy_report":         {"chunk_size": 1000, "overlap": 150},
    "policy_roadmap":        {"chunk_size": 1000, "overlap": 150},
    "investor_presentation": {"chunk_size": 1000, "overlap": 150},
    "market_report":         {"chunk_size": 1000, "overlap": 150},
    "country_overview":      {"chunk_size": 1000, "overlap": 150},
    "industry_analysis":     {"chunk_size": 1000, "overlap": 150},
    "policy_brief":          {"chunk_size": 1000, "overlap": 150},
    "legal_analysis":        {"chunk_size": 1000, "overlap": 150},

    # HTML / webpage
    "corporate_webpage":     {"chunk_size": 500, "overlap": 50},
    "reference_overview":    {"chunk_size": 500, "overlap": 50},
    "industry_pricing":      {"chunk_size": 500, "overlap": 50},
    "market_benchmark":      {"chunk_size": 500, "overlap": 50},
    "policy_announcement":   {"chunk_size": 500, "overlap": 50},
    "industry_media":        {"chunk_size": 500, "overlap": 50},

    # Default fallback
    "default":               {"chunk_size": 800, "overlap": 100},
}

def get_chunk_config(doc_type):
    return CHUNK_CONFIG.get(doc_type, CHUNK_CONFIG["default"])

print("✅ Chunk config loaded")
print(f"Total doc_types defined: {len(CHUNK_CONFIG) - 1}")

✅ Chunk config loaded
Total doc_types defined: 23


In [7]:
from pathlib import Path

# Map part names to data folders
PART_TO_FOLDER = {
    "part1_cost_economics": "dim1_cost",
    "part2_grid_access":    "dim2_grid",
    "part3_policy":         "dim3_policy",
    "part4_utility":        "dim4_utility",
    "part5_approval":       "dim5_approval",
    "part6_unknown":        "dim6_unknown",
}

def get_file_path(part_name, filename):
    folder = PART_TO_FOLDER.get(part_name, "")
    return os.path.join(DATA_DIR, folder, filename)

# Verify all files exist
print("Checking all files...\n")

missing = []
found = []

for part_name, part_data in metadata_master.items():
    # Handle both list and dict structures
    if isinstance(part_data, list):
        files = part_data
    elif isinstance(part_data, dict) and "files" in part_data:
        files = part_data["files"]
    else:
        continue

    for doc in files:
        filename = doc.get("filename", "")
        filepath = get_file_path(part_name, filename)

        if os.path.exists(filepath):
            found.append(filename)
        else:
            missing.append((part_name, filename, filepath))

print(f"Found:   {len(found)} files")
print(f"Missing: {len(missing)} files")

if missing:
    print("\nMissing files:")
    for part, name, path in missing:
        print(f"  [{part}] {name}")
        print(f"    Expected: {path}")

Checking all files...

Found:   41 files
Missing: 0 files


In [11]:
import fitz  # PyMuPDF
from bs4 import BeautifulSoup
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document

def extract_text_from_pdf(filepath):
    """Extract text from PDF using PyMuPDF"""
    doc = fitz.open(filepath)
    text = ""
    for page in doc:
        text += page.get_text()
    doc.close()
    return text

def extract_text_from_html(filepath):
    """Extract text from HTML using BeautifulSoup"""
    with open(filepath, "r", encoding="utf-8", errors="ignore") as f:
        soup = BeautifulSoup(f, "html.parser")
    
    # Remove script and style tags
    for tag in soup(["script", "style", "nav", "footer", "header"]):
        tag.decompose()
    
    return soup.get_text(separator="\n", strip=True)

def build_metadata_prefix(doc_meta):
    """Build metadata prefix string to prepend to every chunk"""
    rag = doc_meta.get("rag", {})
    source = doc_meta.get("source", {})
    content = doc_meta.get("content", {})
    
    prefix = f"""[SOURCE: {source.get('organization', 'Unknown')}]
[DIMENSION: {rag.get('dimension', 'unknown')}]
[REGION: {rag.get('region', 'unknown')}]
[STATUS: {rag.get('status', 'unknown')}]
[PROGRAMME: {content.get('programme', 'N/A')}]
[YEAR: {source.get('date_published', 'unknown')}]
---
"""
    return prefix

def build_flat_metadata(doc_meta):
    """Build flat metadata dict for Chroma (no nested objects)"""
    rag = doc_meta.get("rag", {})
    source = doc_meta.get("source", {})
    content = doc_meta.get("content", {})
    
    dimension = rag.get("dimension", "unknown")
    region = rag.get("region", "unknown")
    
    return {
        "file_id":              doc_meta.get("file_id", ""),
        "filename":             doc_meta.get("filename", ""),
        "organization":         source.get("organization", ""),
        "date_published":       source.get("date_published", ""),
        "title":                content.get("title", ""),
        "doc_type":             content.get("doc_type", ""),
        "dimension":            ", ".join(dimension) if isinstance(dimension, list) else str(dimension),
        "region":               ", ".join(region) if isinstance(region, list) else str(region),
        "status":               rag.get("status", "unknown"),
        "priority":             rag.get("priority", "medium"),
        "doc_role":             rag.get("doc_role", ""),
        "unknown_unknown_flag": str(rag.get("unknown_unknown_flag", False)),
        "warning":              rag.get("warning") or "",
        "alert_message":        rag.get("alert_message") or "",
    }

print("Helper functions defined")

✅ Helper functions defined


In [12]:
def process_document(doc_meta, part_name):
    """Process a single document into chunks with metadata"""
    
    filename = doc_meta.get("filename", "")
    filepath = get_file_path(part_name, filename)
    content = doc_meta.get("content", {})
    doc_type = content.get("doc_type", "default")
    
    # Get chunk config based on doc_type
    config = get_chunk_config(doc_type)
    chunk_size = config["chunk_size"]
    overlap = config["overlap"]
    
    # Build metadata
    flat_meta = build_flat_metadata(doc_meta)
    prefix = build_metadata_prefix(doc_meta)
    
    all_chunks = []
    
    # --- 1. Summary chunk ---
    summary = content.get("summary", "")
    if summary:
        summary_doc = Document(
            page_content=prefix + "[CHUNK_TYPE: summary]\n" + summary,
            metadata={**flat_meta, "chunk_type": "summary"}
        )
        all_chunks.append(summary_doc)
    
    # --- 2. Key data chunk ---
    key_data = content.get("key_data", {})
    if key_data:
        key_data_text = "\n".join([f"{k}: {v}" for k, v in key_data.items()])
        key_data_doc = Document(
            page_content=prefix + "[CHUNK_TYPE: key_data]\n" + key_data_text,
            metadata={**flat_meta, "chunk_type": "key_data"}
        )
        all_chunks.append(key_data_doc)
    
    # --- 3. Body chunks (skip PNG) ---
    if filename.lower().endswith(".png"):
        print(f"  ⏭️  Skipped (PNG): {filename}")
        return all_chunks
    
    # Extract text based on file type
    try:
        if filename.lower().endswith(".pdf"):
            raw_text = extract_text_from_pdf(filepath)
        elif filename.lower().endswith(".html"):
            raw_text = extract_text_from_html(filepath)
        else:
            print(f"  Unknown format: {filename}")
            return all_chunks
    except Exception as e:
        print(f"  Failed to extract: {filename} — {e}")
        return all_chunks
    
    # Split into chunks
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size,
        chunk_overlap=overlap,
        separators=["\n\n", "\n", ".", " "]
    )
    
    body_chunks = splitter.create_documents(
        texts=[prefix + "[CHUNK_TYPE: body]\n" + raw_text],
        metadatas=[{**flat_meta, "chunk_type": "body"}]
    )
    
    all_chunks.extend(body_chunks)
    return all_chunks


print("process_document function defined")

process_document function defined


In [15]:
def process_part(part_name, part_data):
    """Process all documents in a single part"""
    
    # Handle both list and dict structures
    if isinstance(part_data, list):
        files = part_data
    elif isinstance(part_data, dict) and "files" in part_data:
        files = part_data["files"]
    else:
        print(f"  ⚠️  Unrecognised structure for {part_name}")
        return []
    
    all_chunks = []
    
    for doc_meta in files:
        filename = doc_meta.get("filename", "unknown")
        print(f"  Processing: {filename}")
        
        chunks = process_document(doc_meta, part_name)
        all_chunks.extend(chunks)
        print(f"    → {len(chunks)} chunks")
    
    return all_chunks


# Test with part1 only first
print("Testing with part1_cost_economics...\n")
test_chunks = process_part("part1_cost_economics", metadata_master["part1_cost_economics"])
print(f"\n Part 1 total chunks: {len(test_chunks)}")
print(f"Sample chunk preview:\n{test_chunks[0].page_content[:300]}")



Testing with part1_cost_economics...

  Processing: irena_malaysia_energy_transition_2023.pdf
    → 414 chunks
  Processing: mordor_malaysia_solar_market_2026.html
    → 39 chunks
  Processing: progressture_capex_2024.html
    → 26 chunks
  Processing: iea_pvps_malaysia_2025.html
    → 22 chunks

 Part 1 total chunks: 501
Sample chunk preview:
[SOURCE: International Renewable Energy Agency (IRENA)]
[DIMENSION: ['cost', 'utility', 'policy']]
[REGION: ['peninsular', 'sabah', 'sarawak', 'federal']]
[STATUS: active]
[PROGRAMME: LSS / NEM / FiT / NETR]
[YEAR: 2023-03-09]
---
[CHUNK_TYPE: summary]
IRENA's comprehensive Malaysia Energy Transitio


In [17]:
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

# Load embedding model
print("Loading embedding model...")
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True}
)
print(" Embedding model loaded")

# Create Chroma DB with Part 1
print("\nBuilding Chroma DB with Part 1...")
vectorstore = Chroma.from_documents(
    documents=test_chunks,
    embedding=embeddings,
    persist_directory=CHROMA_DIR,
    collection_name="powertrust_malaysia"
)

print(f" Part 1 stored in Chroma DB")
print(f"Total documents in DB: {vectorstore._collection.count()}")

Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/52.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

 Embedding model loaded

Building Chroma DB with Part 1...
 Part 1 stored in Chroma DB
Total documents in DB: 501


In [18]:
# Process and add Part 2 to 6 into the same Chroma DB
parts_to_process = [
    "part2_grid_access",
    "part3_policy",
    "part4_utility",
    "part5_approval",
    "part6_unknown"
]

total_added = 501 

for part_name in parts_to_process:
    print(f"\nProcessing {part_name}...")
    part_data = metadata_master[part_name]
    
    chunks = process_part(part_name, part_data)
    
    if chunks:
        vectorstore.add_documents(chunks)
        total_added += len(chunks)
        print(f" {part_name} done — {len(chunks)} chunks added")
    else:
        print(f" ! {part_name} — no chunks produced")

print(f"\n All parts processed")
print(f"Total documents in DB: {vectorstore._collection.count()}")


Processing part2_grid_access...
  Processing: tnb_malaysian_grid_code_2020.pdf
    → 1242 chunks
  Processing: tnb_dg_interconnection_guideline_2018.pdf
    → 230 chunks
  Processing: tnb_investor_deck_2025.pdf
    → 27 chunks
  Processing: wikipedia_malaysia_electricity.html
    → 23 chunks
  Processing: tnb_der_integration.html
    → 130 chunks
  Processing: tnb_data_analytics.html
    → 203 chunks
  Processing: seda_atap_guidelines_2025.pdf
    → 100 chunks
  Processing: atap_assessment_study_form.pdf
    → 12 chunks
  Processing: seda_atap_pss_process_flow.pdf
    → 6 chunks
  Processing: seda_atap_ccc_cas_process_flow.pdf
    → 4 chunks
  Processing: renewablewatch_malaysia_grid_2024.html
    → 23 chunks
  Processing: non_domestic_application_process.png
  ⏭️  Skipped (PNG): non_domestic_application_process.png
    → 2 chunks
 part2_grid_access done — 2002 chunks added

Processing part3_policy...
  Processing: seda_nem_rakyat_gomen_nov2024.pdf
    → 266 chunks
  Processing: seda_

In [19]:
# Test query
test_query = "What is the LCOE for solar energy in Malaysia?"

results = vectorstore.similarity_search(
    query=test_query,
    k=3
)

print(f"Query: {test_query}\n")
print(f"Top {len(results)} results:\n")

for i, doc in enumerate(results):
    print(f"--- Result {i+1} ---")
    print(f"Source:    {doc.metadata.get('organization', 'unknown')}")
    print(f"Dimension: {doc.metadata.get('dimension', 'unknown')}")
    print(f"Region:    {doc.metadata.get('region', 'unknown')}")
    print(f"Status:    {doc.metadata.get('status', 'unknown')}")
    print(f"Content:   {doc.page_content[:200]}")
    print()
    

Query: What is the LCOE for solar energy in Malaysia?

Top 3 results:

--- Result 1 ---
Source:    BloombergNEF
Dimension: cost, utility
Region:    peninsular, federal
Status:    active
Content:   mounted utility-scale solar projects in Peninsular Malaysia compared to in the Eastern Malaysian states of 
Sarawak and Sabah, the solar and solar-with-storage LCOE data shown in this report mainly re

--- Result 2 ---
Source:    SEDA Malaysia / Energy Commission (ST)
Dimension: grid
Region:    peninsular
Status:    active
Content:   [SOURCE: SEDA Malaysia / Energy Commission (ST)]
[DIMENSION: ['grid']]
[REGION: ['peninsular']]
[STATUS: active]
[PROGRAMME: Solar ATAP]
[YEAR: 2025-12-31]
---
[CHUNK_TYPE: key_data]
ccc_applies_to: d

--- Result 3 ---
Source:    BloombergNEF
Dimension: cost, utility
Region:    peninsular, federal
Status:    active
Content:   combined-cycle gas turbines (CCGTs) for hydrogen blending (Figure 1 and Figure 2). 
• 
Utility-scale solar is already the cheapest source o

In [21]:
def query_with_filter(query, region=None, k=3):
    """Query with metadata filtering and proactive alert check"""
    
    # Build filter
    where_filter = {"status": "active"}
    if region:
        where_filter["region"] = {"$contains": region}
    
    # Semantic search with filter
    try:
        results = vectorstore.similarity_search(
            query=query,
            k=k,
            filter=where_filter
        )
    except Exception:
        # Fallback without filter if filter fails
        results = vectorstore.similarity_search(query=query, k=k)
    
    # Check for proactive alerts
    alerts = []
    # all_results = vectorstore.similarity_search(query=query, k=10)
    all_results = vectorstore.similarity_search(query=query, k=50)

    for doc in all_results:
        if doc.metadata.get("unknown_unknown_flag") == "True":
            alert = doc.metadata.get("alert_message", "")
            if alert and alert not in alerts:
                alerts.append(alert)
    
    return results, alerts


# Test
query = "What are the costs to build a solar farm in Malaysia?"
region = "peninsular"

results, alerts = query_with_filter(query, region=region)

print(f"Query: {query}")
print(f"Region filter: {region}\n")

print("--- Results ---")
for i, doc in enumerate(results):
    print(f"\nResult {i+1}")
    print(f"Source:    {doc.metadata.get('organization')}")
    print(f"Dimension: {doc.metadata.get('dimension')}")
    print(f"Content:   {doc.page_content[:200]}")

print("\n--- Proactive Alerts ---")
if alerts:
    for alert in alerts:
        print(f"⚠️  {alert}\n")
else:
    print("No alerts triggered")

Query: What are the costs to build a solar farm in Malaysia?
Region filter: peninsular

--- Results ---

Result 1
Source:    Progressture Solar
Dimension: cost
Content:   If a business wanted to install 1MWp (1,000kWp) of solar panels on a warehouse with an area of about 8,900sqm, the total cost of solar panel installation would be around RM1.8 million. Although this c

Result 2
Source:    Progressture Solar
Dimension: cost
Content:   RM14,000 to RM46,000
. This price range varies depending on the type of house and roof size, according to the
Malay Mail
[1]. Specifically, the following factors will impact the pricing of solar insta

Result 3
Source:    International Renewable Energy Agency (IRENA)
Dimension: cost, utility, policy
Content:   Peninsular Malaysia
Sabah
Sarawak
Bioenergy resource potential (MW)
Municipal solid waste
Biomass
Biogas
50 | MALAYSIA ENERGY TRANSITION OUTLOOK
of around USD 2.5/kWh for hydropower with capacity fact

--- Proactive Alerts ---
⚠️  Did you know? Data

In [23]:
# Persist the vectorstore to disk
vectorstore.persist()
print(f"Vectorstore persisted to: {CHROMA_DIR}")
print(f"Total documents: {vectorstore._collection.count()}")

Vectorstore persisted to: /Users/bella/powertrust_rag/chroma_db
Total documents: 6342
